## 실습 1. 개선 데이터 확인하기
먼저 모델을 바꾸기 전에 데이터를 확인합니다.


In [14]:
# =========================================================
# Chapter 07. 실행되는 모델에서 검증된 모델로
# 한국어 텍스트 분류·추천 개선 — 하나의 셀 통합 버전
# =========================================================
# 요구 파일: books_improved.csv (컬럼: 상품명, 분야 포함)
# 필요 패키지: pandas, scikit-learn, kiwipiepy
#   pip install pandas scikit-learn kiwipiepy
# =========================================================

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, f1_score, classification_report
from sklearn.metrics.pairwise import cosine_similarity
from kiwipiepy import Kiwi

# ---------------------------------------------------------
# 1. 데이터 로드 및 확인
# ---------------------------------------------------------
df = pd.read_csv("books_improved.csv", encoding="utf-8-sig")
print("=" * 60)
print("1. 데이터 확인")
print("=" * 60)
print("shape:", df.shape)
print(df.columns.tolist())
print(df["분야"].value_counts())
print("결측치:\n", df.isna().sum())
print("중복 상품명:", df["상품명"].duplicated().sum())

# ---------------------------------------------------------
# 2. Train / Test 분리 (Baseline과 Improved가 동일 조건 사용)
# ---------------------------------------------------------
X = df["상품명"]
y = df["분야"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# ---------------------------------------------------------
# 3. Baseline 모델: 원본 제목 -> TF-IDF -> Naive Bayes
# ---------------------------------------------------------
print("\n" + "=" * 60)
print("2. Baseline 측정")
print("=" * 60)

baseline_vectorizer = TfidfVectorizer()
X_train_vec = baseline_vectorizer.fit_transform(X_train)
X_test_vec = baseline_vectorizer.transform(X_test)

baseline_model = MultinomialNB()
baseline_model.fit(X_train_vec, y_train)
baseline_pred = baseline_model.predict(X_test_vec)

baseline_accuracy = accuracy_score(y_test, baseline_pred)
baseline_f1 = f1_score(y_test, baseline_pred, average="macro")
baseline_correct = int((baseline_pred == y_test.values).sum())

print(f"Baseline Accuracy: {baseline_accuracy:.4f}")
print(f"Baseline Macro F1: {baseline_f1:.4f}")
print(f"정답: {baseline_correct} / {len(y_test)}")
print(classification_report(y_test, baseline_pred))

# ---------------------------------------------------------
# 4. 한국어 형태소 분석 + 단어 필터링 함수 정의
# ---------------------------------------------------------
kiwi = Kiwi()

# 최소한의 불용어만 적용 (분야 구분에 도움이 되는 단어는 제거하지 않음)
STOPWORDS = {"에디션"}
KEEP_TAGS = {"NNG", "NNP", "SL"}  # 일반명사, 고유명사, 영문


def tokenize_and_filter(text: str) -> str:
    """형태소 분석 -> 핵심 품사 추출 -> 짧은 토큰/불용어 필터링."""
    if not isinstance(text, str):
        return ""
    tokens = []
    for token in kiwi.tokenize(text):
        if token.tag not in KEEP_TAGS:
            continue
        form = token.form.strip()
        if len(form) < 2:  # 너무 짧은 토큰 제외
            continue
        if form.isdigit():  # 숫자로만 구성된 토큰 제외
            continue
        if form in STOPWORDS:  # 불용어 제외
            continue
        tokens.append(form)
    return " ".join(tokens)


# 전체 데이터에 전처리 적용 (확인용 컬럼 추가)
df["상품명_정제"] = df["상품명"].apply(tokenize_and_filter)
print("\n" + "=" * 60)
print("3. 전처리 전/후 비교 (샘플 10건)")
print("=" * 60)
print(df[["상품명", "상품명_정제"]].head(10))

# Baseline과 동일한 split index를 재사용하기 위해 정제 텍스트를 매핑
train_texts = X_train.apply(tokenize_and_filter)
test_texts = X_test.apply(tokenize_and_filter)

# ---------------------------------------------------------
# 5. Improved 모델: 형태소 분석 + 필터링 -> TF-IDF -> Naive Bayes
# ---------------------------------------------------------
print("\n" + "=" * 60)
print("4. Improved 모델 평가")
print("=" * 60)

improved_vectorizer = TfidfVectorizer()
X_train_improved = improved_vectorizer.fit_transform(train_texts)
X_test_improved = improved_vectorizer.transform(test_texts)

improved_model = MultinomialNB()
improved_model.fit(X_train_improved, y_train)
improved_pred = improved_model.predict(X_test_improved)

improved_accuracy = accuracy_score(y_test, improved_pred)
improved_f1 = f1_score(y_test, improved_pred, average="macro")
improved_correct = int((improved_pred == y_test.values).sum())

print(f"Improved Accuracy: {improved_accuracy:.4f}")
print(f"Improved Macro F1: {improved_f1:.4f}")
print(f"정답: {improved_correct} / {len(y_test)}")
print(classification_report(y_test, improved_pred))

# ---------------------------------------------------------
# 6. Before / After 요약표
# ---------------------------------------------------------
print("\n" + "=" * 60)
print("5. Before / After 요약")
print("=" * 60)
summary = pd.DataFrame(
    {
        "Model": ["Baseline", "Improved"],
        "Accuracy": [baseline_accuracy, improved_accuracy],
        "Macro F1": [baseline_f1, improved_f1],
        "정답": [f"{baseline_correct} / {len(y_test)}", f"{improved_correct} / {len(y_test)}"],
    }
)
print(summary.to_string(index=False))
print(f"정확도 변화: {(improved_accuracy - baseline_accuracy) * 100:.2f}%p")

# ---------------------------------------------------------
# 7. 추천 로직 개선: 같은 분야 후보 + 유사도 0 제외
# ---------------------------------------------------------
print("\n" + "=" * 60)
print("6. 추천 로직 (Before / After)")
print("=" * 60)

# 전체 데이터 기준 TF-IDF (추천용 벡터라이저는 별도로 전체 데이터에 fit)
full_vectorizer_baseline = TfidfVectorizer()
full_matrix_baseline = full_vectorizer_baseline.fit_transform(df["상품명"])

full_vectorizer_improved = TfidfVectorizer()
full_matrix_improved = full_vectorizer_improved.fit_transform(df["상품명_정제"])


def recommend_baseline(selected_index: int, top_n: int = 5):
    """기존 방식: 유사도 0이어도 무조건 Top N 반환."""
    sims = cosine_similarity(
        full_matrix_baseline[selected_index], full_matrix_baseline
    ).flatten()
    sims[selected_index] = -1  # 자기 자신 제외
    top_idx = sims.argsort()[::-1][:top_n]
    return df.iloc[top_idx][["상품명", "분야"]].assign(similarity=sims[top_idx])


def recommend_improved(selected_index: int, top_n: int = 5):
    """개선 방식: 같은 분야 후보 + 유사도 0 제외."""
    selected_category = df.loc[selected_index, "분야"]
    candidate_df = df[df["분야"] == selected_category].copy()
    candidate_positions = candidate_df.index.tolist()

    sims = cosine_similarity(
        full_matrix_improved[selected_index], full_matrix_improved[candidate_positions]
    ).flatten()
    candidate_df["similarity"] = sims

    # 자기 자신 제외 + 유사도 0 초과만 필터링
    candidate_df = candidate_df[candidate_df.index != selected_index]
    candidate_df = candidate_df[candidate_df["similarity"] > 0]
    candidate_df = candidate_df.sort_values("similarity", ascending=False)

    if candidate_df.empty:
        return "현재 기준으로 유사도가 있는 추천 도서를 찾지 못했습니다."
    return candidate_df.head(top_n)[["상품명", "분야", "similarity"]]


# 데이터 첫 번째 도서를 예시로 추천 결과 비교
sample_idx = 0
print(f"\n[선택 도서] {df.loc[sample_idx, '상품명']} (분야: {df.loc[sample_idx, '분야']})")

print("\n-- Baseline 추천 (유사도 0 포함 가능) --")
print(recommend_baseline(sample_idx))

print("\n-- Improved 추천 (같은 분야 + 유사도 0 제외) --")
print(recommend_improved(sample_idx))

# ---------------------------------------------------------
# 8. 전체 도서 기준 "추천 없음" 비율 확인
# ---------------------------------------------------------
no_reco_count = 0
for idx in df.index:
    result = recommend_improved(idx)
    if isinstance(result, str):
        no_reco_count += 1

print("\n" + "=" * 60)
print("7. 추천 불가 도서 비율")
print("=" * 60)
print(f"추천 없음: {no_reco_count} / {len(df)} ({no_reco_count / len(df) * 100:.1f}%)")

print("\n실습 완료.")

1. 데이터 확인
shape: (4316, 12)
['상품코드', '판매상품 ID', '상품명', '정가', '판매가', '할인율', '적립율', '적립예정포인트', '인물', '출판사', '발행(출시)일자', '분야']
분야
소설        1000
시/에세이     1000
인문        1000
컴퓨터/IT     645
건강         267
가정/육아      216
요리         188
Name: count, dtype: int64
결측치:
 상품코드        0
판매상품 ID     0
상품명         0
정가          0
판매가         0
할인율         0
적립율         0
적립예정포인트     0
인물          0
출판사         0
발행(출시)일자    0
분야          0
dtype: int64
중복 상품명: 50

2. Baseline 측정
Baseline Accuracy: 0.5752
Baseline Macro F1: 0.4268
정답: 497 / 864
              precision    recall  f1-score   support

       가정/육아       0.00      0.00      0.00        43
          건강       1.00      0.13      0.23        54
          소설       0.46      0.83      0.59       200
       시/에세이       0.58      0.53      0.55       200
          요리       1.00      0.13      0.23        38
          인문       0.61      0.57      0.59       200
      컴퓨터/IT       0.82      0.76      0.79       129

    accuracy                

c:\dev\python_basics\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\dev\python_basics\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\dev\python_basics\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])



3. 전처리 전/후 비교 (샘플 10건)
                 상품명             상품명_정제
0               싯다르타               싯다르타
1  한국사 이상현상 연구원(일반판)  한국사 이상 현상 연구원 일반판
2        빵충 사육 준수 사항           사육 준수 사항
3                 테오                 테오
4                수족관                수족관
5         녹색 절벽의 신자들           녹색 절벽 신자
6                 모순                 모순
7             서리꽃 세트              서리 세트
8                데미안                데미안
9         브람스를 좋아하세요                브람스

4. Improved 모델 평가
Improved Accuracy: 0.5845
Improved Macro F1: 0.5054
정답: 505 / 864
              precision    recall  f1-score   support

       가정/육아       0.77      0.23      0.36        43
          건강       1.00      0.20      0.34        54
          소설       0.51      0.79      0.62       200
       시/에세이       0.48      0.46      0.47       200
          요리       1.00      0.21      0.35        38
          인문       0.61      0.65      0.62       200
      컴퓨터/IT       0.80      0.75      0.78       129

    accuracy     

## 실행 결과 요약

### 1. 데이터 현황
- 총 **4,316건**, 7개 분야
- 소설 / 시·에세이 / 인문 각 1,000건, 컴퓨터·IT 645건, 건강 267건, 가정·육아 216건, 요리 188건
- 결측치 없음, 중복 상품명 50건

### 2. 분류 성능 (Baseline vs Improved)

| 구분 | Accuracy | Macro F1 | 정답 수 |
|---|---|---|---|
| Baseline | 57.52% | 0.4268 | 497/864 |
| Improved | 58.45% | 0.5054 | 505/864 |
| **변화** | **+0.93%p** | **+0.0786** | **+8건** |

- Accuracy는 큰 변화 없지만 **Macro F1이 크게 상승** → 데이터 불균형 상황에서 소수 분야 성능 개선을 의미
- Baseline은 **가정/육아(f1 0.00), 건강(0.23), 요리(0.23)** 등 소수 분야를 거의 맞히지 못함
- Improved는 형태소 분석으로 이 분야들의 recall·f1이 개선됨 (precision은 1.00 유지, recall만 소폭 상승)
- 반면 **시/에세이는 0.55 → 0.47로 하락** — 형태소 필터링이 감성적·짧은 표현을 걸러내며 소설/인문과 혼동 증가

### 3. 전처리 예시
| 원본 | 정제 후 |
|---|---|
| 브람스를 좋아하세요 | 브람스 |
| 서리꽃 세트 | 서리 세트 |
| 빵충 사육 준수 사항 | 사육 준수 사항 (신조어 "빵충" 손실) |

### 4. 추천 시스템
- "싯다르타" 사례: Baseline·Improved 모두 동일 계열(완역본, 큰글자책 등) 정상 추천
- 해당 사례는 유사도 0인 케이스가 없어 두 방식 차이가 뚜렷하게 드러나지 않음

### 총평
- **Macro F1 · 분야별 recall/precision 균형** 기준으로는 개선 효과 확인됨
- 소수 분야(요리/건강/가정육아)는 여전히 recall이 낮아 **데이터 불균형이 근본 한계**
- 다음 단계: 소수 분야 오버샘플링 또는 `class_weight` 조정 검토 필요

## 실습 2. Baseline을 먼저 측정하기


In [15]:
# ============================================================
# 실습 2. Baseline 측정하기
# 도서 제목 -> TF-IDF -> Multinomial Naive Bayes -> 분야 예측
# ============================================================

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report

# 데이터 로드 (상품명, 분야 컬럼 포함된 CSV)
df = pd.read_csv("books_improved.csv", encoding="utf-8-sig")

# Train / Test 분리
X = df["상품명"]
y = df["분야"]
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

# TF-IDF 벡터화 (훈련 데이터에만 fit, 테스트는 transform만)
baseline_vectorizer = TfidfVectorizer()
X_train_vec = baseline_vectorizer.fit_transform(X_train)
X_test_vec = baseline_vectorizer.transform(X_test)

# Naive Bayes 모델 학습 및 예측
baseline_model = MultinomialNB()
baseline_model.fit(X_train_vec, y_train)
baseline_pred = baseline_model.predict(X_test_vec)

# 평가
baseline_accuracy = accuracy_score(y_test, baseline_pred)
print("Baseline Accuracy:", baseline_accuracy)
print(classification_report(y_test, baseline_pred))

Baseline Accuracy: 0.5752314814814815
              precision    recall  f1-score   support

       가정/육아       0.00      0.00      0.00        43
          건강       1.00      0.13      0.23        54
          소설       0.46      0.83      0.59       200
       시/에세이       0.58      0.53      0.55       200
          요리       1.00      0.13      0.23        38
          인문       0.61      0.57      0.59       200
      컴퓨터/IT       0.82      0.76      0.79       129

    accuracy                           0.58       864
   macro avg       0.64      0.42      0.43       864
weighted avg       0.61      0.58      0.54       864



c:\dev\python_basics\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\dev\python_basics\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\dev\python_basics\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


## Baseline 측정 결과 요약

**전체 성능**
- Accuracy: **57.52%** (864건 중 약 497건 정답)
- Macro F1: **0.43** (Accuracy보다 훨씬 낮음 → 분야별 성능 편차가 크다는 신호)

**분야별 성능**

| 분야 | Precision | Recall | F1 | 해석 |
|---|---|---|---|---|
| 가정/육아 | 0.00 | 0.00 | 0.00 | 전혀 못 맞춤 — 한 건도 이 분야로 예측하지 않음 |
| 건강 | 1.00 | 0.13 | 0.23 | 예측하면 항상 맞지만, 대부분 놓침(과소예측) |
| 요리 | 1.00 | 0.13 | 0.23 | 건강과 동일 패턴 |
| 소설 | 0.46 | 0.83 | 0.59 | recall은 높지만 precision 낮음 → 다른 분야를 소설로 과다 예측 |
| 시/에세이 | 0.58 | 0.53 | 0.55 | 중간 수준 |
| 인문 | 0.61 | 0.57 | 0.59 | 중간 수준 |
| 컴퓨터/IT | 0.82 | 0.76 | 0.79 | 가장 우수 — "파이썬", "SQL" 등 고유 키워드로 구분 용이 |

**핵심 문제**
1. **데이터 불균형의 영향**: 소설/시에세이/인문(각 200건)이 건강/요리/가정육아(38~54건)보다 훨씬 많아, 애매한 제목은 다수 클래스(특히 소설)로 쏠려 예측되는 경향
2. **가정/육아가 완전히 무시됨**: recall 0.00 → 이 분야 패턴을 전혀 학습하지 못함
3. Accuracy(0.58)만 보면 준수해 보이지만, Macro F1(0.43)을 보면 소수 분야는 사실상 작동하지 않는 모델임이 드러남

**경고 메시지**: `UndefinedMetricWarning`은 가정/육아가 0건 예측되어 precision 분모가 0이 된 데서 발생

**다음 단계**: Improved 모델(형태소 분석 + 필터링)과 비교해 Macro F1 개선 여부 및 소수 분야 recall 변화를 확인

## 실습 3. 한국어 형태소 분석 적용하기

 

기본 TF-IDF는 제목 문자열을 그대로 사용합니다.

 

이번에는 한국어 제목에서 의미 있는 형태소를 추출해 TF-IDF 입력을 바꿔 봅니다.

 

예를 들어 다음 제목을 생각해 봅니다.


In [16]:
# ============================================================
# 실습 3. 한국어 형태소 분석 적용하기
# Kiwi로 명사(NNG)·고유명사(NNP)·영문(SL) 토큰만 추출
# ============================================================

from kiwipiepy import Kiwi
import pandas as pd

kiwi = Kiwi()

# --- 형태소 확인 (샘플) ---
sample = "처음 배우는 파이썬 데이터 분석"
for token in kiwi.tokenize(sample):
    print(token.form, token.tag)

# --- 핵심 품사만 추출하는 함수 정의 ---
KEEP_TAGS = {"NNG", "NNP", "SL"}  # 일반명사, 고유명사, 영문

def extract_tokens(text: str) -> str:
    if not isinstance(text, str):
        return ""
    return " ".join(
        token.form for token in kiwi.tokenize(text) if token.tag in KEEP_TAGS
    )

# --- 전체 상품명에 적용 ---
df = pd.read_csv("books_improved.csv", encoding="utf-8-sig")
df["상품명_토큰"] = df["상품명"].apply(extract_tokens)

# --- 전처리 전후 확인 ---
print(df[["상품명", "상품명_토큰"]].head(10))

처음 NNG
배우 VV
는 ETM
파이썬 NNP
데이터 NNG
분석 NNG
                 상품명             상품명_토큰
0               싯다르타               싯다르타
1  한국사 이상현상 연구원(일반판)  한국사 이상 현상 연구원 일반판
2        빵충 사육 준수 사항       빵 충 사육 준수 사항
3                 테오                 테오
4                수족관                수족관
5         녹색 절벽의 신자들           녹색 절벽 신자
6                 모순                 모순
7             서리꽃 세트            서리 꽃 세트
8                데미안                데미안
9         브람스를 좋아하세요                브람스


## 형태소 분석 결과 요약

### 1. 샘플 토큰화 결과
"처음 배우는 파이썬 데이터 분석" 분석 시:

| 토큰 | 품사 | 비고 |
|---|---|---|
| 처음 | NNG | 일반명사 → 유지 |
| 배우 | VV | 동사 → **제외** |
| 는 | ETM | 관형형 어미 → **제외** |
| 파이썬 | NNP | 고유명사 → 유지 |
| 데이터 | NNG | 일반명사 → 유지 |
| 분석 | NNG | 일반명사 → 유지 |

→ 최종 토큰: **"처음 파이썬 데이터 분석"**

### 2. 전체 상품명 전처리 결과

| 원본 | 정제 후 | 패턴 |
|---|---|---|
| 싯다르타 | 싯다르타 | 단일 명사, 변화 없음 |
| 한국사 이상현상 연구원(일반판) | 한국사 이상 현상 연구원 일반판 | 괄호 제거, "이상현상"이 "이상+현상"으로 **분리** |
| 빵충 사육 준수 사항 | 빵 충 사육 준수 사항 | "빵충"이 "빵+충"으로 **분리** (신조어 의미 손실) |
| 테오 | 테오 | 변화 없음 |
| 수족관 | 수족관 | 변화 없음 |
| 녹색 절벽의 신자들 | 녹색 절벽 신자 | 조사 "의", 접미사 "들" 제거 |
| 모순 | 모순 | 변화 없음 |
| 서리꽃 세트 | 서리 꽃 세트 | "서리꽃"이 "서리+꽃"으로 **분리** |
| 데미안 | 데미안 | 변화 없음 |
| 브람스를 좋아하세요 | 브람스 | 조사·동사 제거로 **제목 대부분 소실** |

### 3. 핵심 관찰

**정상적으로 작동한 케이스**
- 조사(를, 의)·어미(들, 는)가 깔끔하게 제거되어 핵심 명사만 남음
- 예: "녹색 절벽의 신자들" → "녹색 절벽 신자"

**부작용: 복합어·신조어 분리 문제**
- 형태소 분석기가 사전에 없는 복합명사나 신조어를 지나치게 잘게 쪼개는 경향 발견
- "이상현상" → "이상 현상", "빵충" → "빵 충", "서리꽃" → "서리 꽃"
- 특히 **"빵충"은 원저작물 특유의 신조어**인데 분리되면서 원래 의미가 완전히 사라짐

**정보 손실 케이스**
- "브람스를 좋아하세요"는 조사·동사가 모두 제거되며 제목의 핵심 정보 대부분이 소실되고 "브람스"만 남음
- 문장형 제목일수록 정보 손실 위험이 큼

### 4. 시사점
형태소 분석이 **조사·어미 제거에는 효과적**이지만, **한국어 고유의 복합어·신조어 처리에는 한계**가 있음을 보여줌. 이는 다음 단계(Improved 모델 평가)에서 분류 성능에 어떤 영향을 미치는지 실제 지표로 확인이 필요함.

In [17]:
# ============================================================
# 실습 4. 불필요한 단어 필터링하기
# 짧은 토큰 / 숫자 토큰 / 최소 불용어만 제거
# ============================================================

# 최소한의 불용어만 적용 (분야 핵심 단어는 절대 제거하지 않음)
stopwords = {
    "에디션",
}

def filter_tokens(token_str: str) -> str:
    """형태소 분석 결과에서 짧은 토큰·숫자 토큰·불용어를 제거."""
    if not isinstance(token_str, str) or not token_str:
        return ""
    tokens = token_str.split()
    filtered = [
        t for t in tokens
        if len(t) >= 2          # 너무 짧은 토큰 제외
        and not t.isdigit()     # 숫자로만 구성된 토큰 제외
        and t not in stopwords  # 최소 불용어 제외
    ]
    return " ".join(filtered)

# --- 형태소 토큰(상품명_토큰)에 필터 적용 ---
df["상품명_정제"] = df["상품명_토큰"].apply(filter_tokens)

# --- 전처리 전후 비교 ---
display(
    df[["상품명", "상품명_토큰", "상품명_정제"]].head(10)
)

,상품명,상품명_토큰,상품명_정제
0,싯다르타,싯다르타,싯다르타
1,한국사 이상현상 연구원(일반판),한국사 이상 현상 연구원 일반판,한국사 이상 현상 연구원 일반판
2,빵충 사육 준수 사항,빵 충 사육 준수 사항,사육 준수 사항
3,테오,테오,테오
4,수족관,수족관,수족관
5,녹색 절벽의 신자들,녹색 절벽 신자,녹색 절벽 신자
6,모순,모순,모순
7,서리꽃 세트,서리 꽃 세트,서리 세트
8,데미안,데미안,데미안
9,브람스를 좋아하세요,브람스,브람스


## 필터링 결과 요약

### 전처리 전후 비교 (상품명 → 상품명_토큰 → 상품명_정제)

| 상품명 | 상품명_토큰 | 상품명_정제 | 변화 |
|---|---|---|---|
| 싯다르타 | 싯다르타 | 싯다르타 | 변화 없음 |
| 한국사 이상현상 연구원(일반판) | 한국사 이상 현상 연구원 일반판 | 한국사 이상 현상 연구원 일반판 | 변화 없음 |
| 빵충 사육 준수 사항 | 빵 충 사육 준수 사항 | **사육 준수 사항** | "빵", "충" 제거(1글자 토큰) |
| 테오 | 테오 | 테오 | 변화 없음 |
| 수족관 | 수족관 | 수족관 | 변화 없음 |
| 녹색 절벽의 신자들 | 녹색 절벽 신자 | 녹색 절벽 신자 | 변화 없음 |
| 모순 | 모순 | 모순 | 변화 없음 |
| 서리꽃 세트 | 서리 꽃 세트 | **서리 세트** | "꽃" 제거(1글자 토큰) |
| 데미안 | 데미안 | 데미안 | 변화 없음 |
| 브람스를 좋아하세요 | 브람스 | 브람스 | 변화 없음 |

### 핵심 관찰

- **10건 중 2건만 변화** — "빵충 사육 준수 사항", "서리꽃 세트"에서만 필터가 작동
- 두 사례 모두 **1글자로 쪼개진 토큰("빵", "충", "꽃")이 제거**된 케이스 → 짧은 토큰 필터(`len < 2`)가 정상 작동
- 이번 샘플에는 숫자 토큰이나 "에디션" 같은 불용어가 없어 해당 필터는 효과가 드러나지 않음
- 실습 3에서 발생했던 "빵충 → 빵 충" 같은 **신조어 분리 문제는 이 단계에서 오히려 완화**됨 (의미 없는 조각이 제거되며 "사육 준수 사항"처럼 더 간결해짐)

### 시사점
불필요한 단어 필터링은 형태소 분석 과정에서 생긴 **의미 없는 짧은 조각을 정리하는 역할**을 함. 다만 이 자체가 분류 성능 향상을 보장하지는 않으므로, 다음 단계(Improved 모델 평가)에서 실제 Accuracy/F1 변화로 효과를 검증해야 함.

## 실습 5. Improved 분류 모델 평가하기

 

이제 형태소 분석과 단어 필터링을 적용한 텍스트로 다시 TF-IDF와 Naive Bayes를 학습합니다.

In [18]:
# ============================================================
# 실습 5. Improved 분류 모델 평가하기 (자기완결형 버전)
# Baseline → 형태소분석+필터링 → Improved 순서로 한 번에 실행
# ============================================================

import pandas as pd
from kiwipiepy import Kiwi
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, f1_score, classification_report

# ---- 데이터 로드 및 Train/Test 분리 ----
df = pd.read_csv("books_improved.csv", encoding="utf-8-sig")
X = df["상품명"]
y = df["분야"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# ---- Baseline: 원본 제목 -> TF-IDF -> Naive Bayes ----
baseline_vectorizer = TfidfVectorizer()
X_train_vec = baseline_vectorizer.fit_transform(X_train)
X_test_vec = baseline_vectorizer.transform(X_test)
baseline_model = MultinomialNB()
baseline_model.fit(X_train_vec, y_train)
baseline_pred = baseline_model.predict(X_test_vec)
baseline_accuracy = accuracy_score(y_test, baseline_pred)

# ---- 형태소 분석 + 필터링 함수 정의 (실습 3, 4 통합) ----
kiwi = Kiwi()
KEEP_TAGS = {"NNG", "NNP", "SL"}  # 일반명사, 고유명사, 영문
stopwords = {"에디션"}

def tokenize(text: str) -> str:
    """형태소 분석 -> 핵심 품사(NNG/NNP/SL)만 추출."""
    if not isinstance(text, str):
        return ""
    return " ".join(t.form for t in kiwi.tokenize(text) if t.tag in KEEP_TAGS)

def filter_tokens(token_str: str) -> str:
    """짧은 토큰 / 숫자 토큰 / 불용어 제거."""
    if not isinstance(token_str, str) or not token_str:
        return ""
    return " ".join(
        t for t in token_str.split()
        if len(t) >= 2 and not t.isdigit() and t not in stopwords
    )

def preprocess(text: str) -> str:
    """tokenize + filter_tokens 결합."""
    return filter_tokens(tokenize(text))

# ---- Improved: 형태소분석+필터링 -> TF-IDF -> Naive Bayes ----
train_texts = X_train.apply(preprocess)
test_texts = X_test.apply(preprocess)

improved_vectorizer = TfidfVectorizer()
X_train_improved = improved_vectorizer.fit_transform(train_texts)
X_test_improved = improved_vectorizer.transform(test_texts)

improved_model = MultinomialNB()
improved_model.fit(X_train_improved, y_train)
improved_pred = improved_model.predict(X_test_improved)
improved_accuracy = accuracy_score(y_test, improved_pred)

# ---- 결과 출력 ----
print("Baseline Accuracy:", baseline_accuracy)
print("Improved Accuracy:", improved_accuracy)
print(classification_report(y_test, improved_pred))

summary = pd.DataFrame({
    "Model": ["Baseline", "Improved"],
    "Accuracy": [baseline_accuracy, improved_accuracy],
    "Macro F1": [
        f1_score(y_test, baseline_pred, average="macro"),
        f1_score(y_test, improved_pred, average="macro"),
    ],
})
print(summary)

# ---- 대표 제목 확인 ----
sample_title = "처음 배우는 파이썬 데이터 분석"
sample_vec = improved_vectorizer.transform([preprocess(sample_title)])
print(f"\n[대표 제목] '{sample_title}' -> 예측 분야:", improved_model.predict(sample_vec)[0])

Baseline Accuracy: 0.5752314814814815
Improved Accuracy: 0.5844907407407407
              precision    recall  f1-score   support

       가정/육아       0.77      0.23      0.36        43
          건강       1.00      0.20      0.34        54
          소설       0.51      0.79      0.62       200
       시/에세이       0.48      0.46      0.47       200
          요리       1.00      0.21      0.35        38
          인문       0.61      0.65      0.62       200
      컴퓨터/IT       0.80      0.75      0.78       129

    accuracy                           0.58       864
   macro avg       0.74      0.47      0.51       864
weighted avg       0.64      0.58      0.57       864

      Model  Accuracy  Macro F1
0  Baseline  0.575231  0.426817
1  Improved  0.584491  0.505427

[대표 제목] '처음 배우는 파이썬 데이터 분석' -> 예측 분야: 컴퓨터/IT


## Improved 모델 평가 결과 요약

### 1. Before / After 비교

| Model | Accuracy | Macro F1 |
|---|---|---|
| Baseline | 57.52% | 0.4268 |
| Improved | 58.45% | 0.5054 |
| **변화** | **+0.93%p** | **+0.0786** |

- Accuracy는 거의 변화가 없지만(+0.93%p), **Macro F1은 뚜렷하게 상승**(+0.0786)
- 이는 소수 분야(가정/육아, 건강, 요리)의 성능이 개선되었기 때문 → 데이터 불균형 상황에서 Accuracy보다 Macro F1이 실제 개선을 더 잘 반영함

### 2. 분야별 성능 변화

| 분야 | Precision | Recall | F1 | 비고 |
|---|---|---|---|---|
| 가정/육아 | 0.77 | 0.23 | 0.36 | Baseline(f1 0.00) 대비 **큰 개선** — 처음으로 유의미하게 예측됨 |
| 건강 | 1.00 | 0.20 | 0.34 | Baseline(0.23) 대비 소폭 개선, recall 여전히 낮음 |
| 요리 | 1.00 | 0.21 | 0.35 | 건강과 유사한 패턴 |
| 소설 | 0.51 | 0.79 | 0.62 | Baseline(0.59) 대비 소폭 상승 |
| 시/에세이 |

In [21]:
# ============================================================
# 실습 6. 추천 결과의 문제 개선하기 (자기완결형 + Fallback 버전)
# Baseline(무조건 Top5) vs Improved(같은 분야 + 유사도>0)
# ============================================================

import pandas as pd
import random
from kiwipiepy import Kiwi
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# ---- 데이터 로드 ----
df = pd.read_csv("books_improved.csv", encoding="utf-8-sig")

# ---- 형태소 분석 + 필터링 함수 (실습 3, 4 통합) ----
kiwi = Kiwi()
KEEP_TAGS = {"NNG", "NNP", "SL"}
stopwords = {"에디션"}

def tokenize(text: str) -> str:
    if not isinstance(text, str):
        return ""
    return " ".join(t.form for t in kiwi.tokenize(text) if t.tag in KEEP_TAGS)

def filter_tokens(token_str: str) -> str:
    if not isinstance(token_str, str) or not token_str:
        return ""
    return " ".join(
        t for t in token_str.split()
        if len(t) >= 2 and not t.isdigit() and t not in stopwords
    )

def preprocess(text: str) -> str:
    return filter_tokens(tokenize(text))

df["상품명_정제"] = df["상품명"].apply(preprocess)

# ---- Baseline 추천용 TF-IDF (원본 제목, 전체 도서 대상) ----
baseline_vectorizer = TfidfVectorizer()
baseline_matrix = baseline_vectorizer.fit_transform(df["상품명"])

def recommend_baseline(selected_index: int, top_n: int = 5):
    """기존 방식: 유사도 0이어도 무조건 Top N 반환."""
    sims = cosine_similarity(baseline_matrix[selected_index], baseline_matrix).flatten()
    sims[selected_index] = -1  # 자기 자신 제외
    top_idx = sims.argsort()[::-1][:top_n]
    result = df.iloc[top_idx][["상품명", "분야"]].copy()
    result["similarity"] = sims[top_idx]
    return result

# ---- Improved 추천용 TF-IDF (정제 텍스트) ----
improved_vectorizer = TfidfVectorizer()
improved_matrix = improved_vectorizer.fit_transform(df["상품명_정제"])

def recommend_improved(selected_index: int, top_n: int = 5):
    """개선 방식: 같은 분야 후보 + 유사도 0 제외."""
    selected_category = df.loc[selected_index, "분야"]
    candidate_df = df[df["분야"] == selected_category].copy()
    candidate_positions = candidate_df.index.tolist()

    sims = cosine_similarity(
        improved_matrix[selected_index], improved_matrix[candidate_positions]
    ).flatten()
    candidate_df["similarity"] = sims

    candidate_df = candidate_df[candidate_df.index != selected_index]  # 자기 자신 제외
    candidate_df = candidate_df[candidate_df["similarity"] > 0]        # 유사도 0 제외
    candidate_df = candidate_df.sort_values("similarity", ascending=False)

    if candidate_df.empty:
        return "현재 기준으로 유사도가 있는 추천 도서를 찾지 못했습니다."
    return candidate_df.head(top_n)[["상품명", "분야", "similarity"]]

# ---- 대표 도서 선택 (원하는 제목으로 변경 가능) ----
target_title = "소년이 온다"  # <- 여기를 원하는 제목으로 바꿔서 테스트 가능

matched = df[df["상품명"].str.contains(target_title, na=False)]

if matched.empty:
    # 지정한 제목이 없으면 데이터 내 임의의 도서로 대체
    random.seed(42)
    selected_index = random.choice(df.index.tolist())
    print(f"'{target_title}'을(를) 찾지 못해 임의의 도서를 선택합니다.")
else:
    selected_index = matched.index[0]

print(f"[선택 도서] {df.loc[selected_index, '상품명']} (분야: {df.loc[selected_index, '분야']})")

print("\n-- Baseline 추천 (유사도 0 포함 가능) --")
print(recommend_baseline(selected_index))

print("\n-- Improved 추천 (같은 분야 + 유사도 0 제외) --")
print(recommend_improved(selected_index))

[선택 도서] 소년이 온다 (분야: 소설)

-- Baseline 추천 (유사도 0 포함 가능) --
                    상품명     분야  similarity
1194        나쁜 소년이 서 있다  시/에세이    0.467395
1586       결국 너의 시간은 온다  시/에세이    0.327293
1845     너무 보고플 땐 눈이 온다  시/에세이    0.319718
1392      후회는 언제나 늦게 온다  시/에세이    0.304105
1492  저녁은 누구에게나 공평하게 온다  시/에세이    0.299528

-- Improved 추천 (같은 분야 + 유사도 0 제외) --
           상품명  분야  similarity
696       소년이로  소설    1.000000
320  바다에서 온 소년  소설    0.722714


## 추천 로직 Before/After 결과 요약

### 선택 도서
**소년이 온다** (분야: 소설)

### Baseline 추천 (전체 도서 대상, 유사도 0 포함 가능)

| 상품명 | 분야 | similarity |
|---|---|---|
| 나쁜 소년이 서 있다 | 시/에세이 | 0.467 |
| 결국 너의 시간은 온다 | 시/에세이 | 0.327 |
| 너무 보고플 땐 눈이 온다 | 시/에세이 | 0.320 |
| 후회는 언제나 늦게 온다 | 시/에세이 | 0.304 |
| 저녁은 누구에게나 공평하게 온다 | 시/에세이 | 0.300 |

### Improved 추천 (같은 분야 + 유사도 0 제외)

| 상품명 | 분야 | similarity |
|---|---|---|
| 소년이로 | 소설 | 1.000 |
| 바다에서 온 소년 | 소설 | 0.723 |

### 핵심 관찰

**Baseline의 문제점이 실제로 드러남**
- Top 5 전체가 **소설이 아닌 시/에세이**로 채워짐 — "소년이 온다"라는 문학 작품에 대한 추천으로는 부적절
- 유사도가 붙은 이유는 단순히 "~다", "온다" 같은 **조사·종결어미가 공유**되었기 때문 (예: "온다", "서 있다", "늦게 온다") → 내용적 유사성이 아니라 문자열 표면 패턴에 의한 착시 매칭
- 즉 "유사도 0"까지는 아니지만, **의미 없는 낮은 유사도(0.3~0.47)가 마치 의미 있는 추천처럼 보이는 사례** — 블로그에서 언급한 "유사도 0 문제"의 확장판이라 할 수 있음

**Improved의 개선 효과**
- 형태소 분석으로 조사·어미가 제거되면서 표면적 어미 매칭이 사라짐
- 같은 분야(소설) 제한 조건 덕분에 시/에세이가 원천적으로 후보에서 배제됨
- 결과적으로 "소년이로", "바다에서 온 소년"처럼 **제목에 "소년"이라는 핵심 명사를 공유하는 진짜 관련 도서만 추천**됨
- 다만 후보가 2권뿐이라 Top5를 채우지 못함 → "억지로 채우지 않는다"는 설계 원칙이 실제로 작동

### 결론
이 사례는 Baseline 추천 로직의 한계를 명확히 보여줌: **표면적 문자열(어미·조사) 유사도는 내용적 관련성과 무관할 수 있다.** 형태소 분석 기반 정제 + 같은 분야 제한 조합이 이런 "가짜 유사도" 문제를 효과적으로 걸러내는 것을 실제 데이터로 확인.